# WeatherPulse: Spark Analysis for 6 Major Indonesian Cities

## Project Overview
**ETS BigData - Anggota 4: Spark Processing Layer**

Analisis mendalam terhadap data cuaca 6 kota besar Indonesia menggunakan Apache Spark untuk menjawab pertanyaan bisnis klien logistik:
- **Pertanyaan Utama:** Kota mana yang kondisi cuacanya paling ekstrem hari ini, dan kapan perkiraan terbaik untuk pengiriman?

### Kota yang Dipantau:
- 🏙️ **JKT** - Jakarta (-6.21, 106.85)
- 🏙️ **SBY** - Surabaya (-7.25, 112.75)
- 🏙️ **SMG** - Semarang (-6.99, 110.42)
- 🏙️ **MDN** - Medan (-3.59, 98.67)
- 🏙️ **MKS** - Makassar (-5.14, 119.41)
- 🏙️ **DPS** - Denpasar (-8.67, 115.21)

In [1]:
# ============================================================================
# SECTION 1: Import Required Libraries
# ============================================================================
# Import PySpark SQL functions, DataFrame operations, and utilities

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, max as spark_max, min as spark_min, count, 
    to_timestamp, hour, date_format, when, round as spark_round,
    desc, asc, collect_list, struct, size, explode, arrays_zip
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType
import pandas as pd
import json
from datetime import datetime
import os

print("✓ All libraries imported successfully")
print(f"Timestamp: {datetime.now()}")

✓ All libraries imported successfully
Timestamp: 2026-04-28 21:25:00.767075


In [2]:
# ============================================================================
# SECTION 2: Initialize SparkSession with HDFS Configuration
# ============================================================================
# Setup Spark untuk koneksi ke HDFS dan processing data cuaca

spark = SparkSession.builder \
    .appName("WeatherPulse-Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Set log level untuk output yang lebih clean
spark.sparkContext.setLogLevel("WARN")

print("✓ SparkSession initialized")
print(f"  App Name: {spark.sparkContext.appName}")
print(f"  Master: {spark.sparkContext.master}")
print(f"  Mode: Local (Development Mode)")

✓ SparkSession initialized
  App Name: WeatherPulse-Analysis
  Master: local[*]
  Mode: Local (Development Mode)


In [3]:
# ============================================================================
# SECTION 3: Load Weather Data (Kafka → HDFS Priority Order - REAL DATA ONLY)
# ============================================================================
# 2 cara untuk mendapatkan data REAL:
# 1. KAFKA LIVE: Baca langsung dari Kafka topics (real-time data dari producer)
# 2. HDFS: Baca dari HDFS yang sudah di-populate oleh consumer
# 3. ERROR: Jika keduanya tidak tersedia (NO SAMPLE DATA - REAL DATA ONLY!)

import os

data_source = None
use_spark_mode = False
df_api_pd = None

# ========== OPSI 1: KAFKA LIVE (Real-time from Producer) ==========
print("📡 OPSI 1: Attempting to read from Kafka topics (weather-api, weather-rss)...")
print("-" * 80)

try:
    from kafka import KafkaConsumer
    import json
    
    KAFKA_BOOTSTRAP = "localhost:9092"
    KAFKA_TIMEOUT = 5000  # 5 second timeout per topic
    
    # Baca dari weather-api topic
    api_data = []
    rss_data = []
    
    try:
        print("  Reading from weather-api topic...")
        consumer_api = KafkaConsumer(
            'weather-api',
            bootstrap_servers=KAFKA_BOOTSTRAP,
            group_id='analysis-group-api',
            auto_offset_reset='earliest',  # Changed to 'earliest' untuk ambil semua data
            consumer_timeout_ms=KAFKA_TIMEOUT,
            value_deserializer=lambda m: json.loads(m.decode('utf-8')) if m else None
        )
        
        for message in consumer_api:
            if message.value:
                api_data.append(message.value)
        
        consumer_api.close()
        print(f"    ✓ Retrieved {len(api_data)} records from weather-api")
        
    except Exception as e:
        print(f"    ✗ Could not read weather-api: {type(e).__name__}")
    
    try:
        print("  Reading from weather-rss topic...")
        consumer_rss = KafkaConsumer(
            'weather-rss',
            bootstrap_servers=KAFKA_BOOTSTRAP,
            group_id='analysis-group-rss',
            auto_offset_reset='earliest',  # Changed to 'earliest' untuk ambil semua data
            consumer_timeout_ms=KAFKA_TIMEOUT,
            value_deserializer=lambda m: json.loads(m.decode('utf-8')) if m else None
        )
        
        for message in consumer_rss:
            if message.value:
                rss_data.append(message.value)
        
        consumer_rss.close()
        print(f"    ✓ Retrieved {len(rss_data)} records from weather-rss")
        
    except Exception as e:
        print(f"    ✗ Could not read weather-rss: {type(e).__name__}")
    
    # Combine API + RSS data
    if len(api_data) > 0 or len(rss_data) > 0:
        combined_data = api_data + rss_data
        
        print(f"\n  📊 Data Structure Debug:")
        print(f"     - API records: {len(api_data)}")
        print(f"     - RSS records: {len(rss_data)}")
        
        if len(api_data) > 0:
            print(f"     - API sample: {list(api_data[0].keys())}")
        if len(rss_data) > 0:
            print(f"     - RSS sample keys: {list(rss_data[0].keys())}")
        
        # Normalize ke format standar (prioritaskan API data karena struktur lengkap)
        normalized_data = []
        
        # Process API data (prioritas utama)
        for record in api_data:
            if isinstance(record, dict):
                # API data harus punya semua field
                if all(k in record for k in ['kode_kota', 'temperature', 'humidity', 'wind_speed', 'timestamp']):
                    normalized_data.append(record)
        
        # Process RSS data (fallback jika API kosong)
        if len(normalized_data) == 0:
            print(f"\n  ⚠️  API data tidak valid atau kosong, attempting RSS fallback...")
            for record in rss_data:
                if isinstance(record, dict):
                    # Try to normalize RSS data - bisa berbeda struktur
                    try:
                        normalized_record = {
                            'kode_kota': record.get('kode_kota', 'RSS'),
                            'nama_kota': record.get('nama_kota', record.get('title', 'Unknown')),
                            'temperature': float(record.get('temperature', 25)),
                            'humidity': float(record.get('humidity', 70)),
                            'wind_speed': float(record.get('wind_speed', 0)),
                            'weather_code': record.get('weather_code', 0),
                            'timestamp': record.get('timestamp', datetime.now().isoformat()),
                            'latitude': record.get('latitude', 0),
                            'longitude': record.get('longitude', 0)
                        }
                        normalized_data.append(normalized_record)
                    except:
                        pass
        
        if len(normalized_data) > 0:
            df_api_pd = pd.DataFrame(normalized_data)
            
            # Validate columns
            required_cols = ['kode_kota', 'temperature', 'humidity', 'wind_speed', 'timestamp']
            if all(col in df_api_pd.columns for col in required_cols):
                data_source = "kafka-live"
                use_spark_mode = False  # Use Pandas mode untuk Kafka data
                print(f"\n✅ KAFKA DATA LOADED: {len(normalized_data)} total records")
                print(f"   Sources: {len(api_data)} API + {len(rss_data)} RSS")
            else:
                print(f"  ⚠ Normalized data struktur tidak sesuai, fallback ke HDFS...")
                print(f"    Available: {df_api_pd.columns.tolist()}")
                df_api_pd = None
        else:
            print(f"  ⚠ No valid/normalizable records from Kafka")

except ImportError:
    print("  ⚠ kafka-python library tidak tersedia, skip Kafka option")
except Exception as e:
    print(f"  ⚠ Kafka connection error: {type(e).__name__}")
    import traceback
    traceback.print_exc()

# ========== OPSI 2: HDFS (Dari Consumer Buffer) ==========
if data_source is None:
    print("\n📁 OPSI 2: Attempting to read from HDFS (consumer buffer)...")
    print("-" * 80)
    
    hdfs_api_path = "hdfs://namenode:8020/data/weather/api/"
    hdfs_rss_path = "hdfs://namenode:8020/data/weather/rss/"
    
    try:
        # Try to read API data from HDFS
        df_api_spark = spark.read \
            .option("multiLine", True) \
            .option("inferSchema", True) \
            .json(hdfs_api_path)
        
        test_count = df_api_spark.limit(1).count()
        if test_count > 0:
            df_api_pd = df_api_spark.toPandas()
            data_source = "hdfs-consumer"
            use_spark_mode = True
            print(f"  ✓ API data from HDFS: {len(df_api_pd)} records")
        else:
            raise Exception("HDFS API path is empty")
            
    except Exception as e:
        print(f"  ⚠ HDFS not available: {type(e).__name__}")

# ========== OPSI 3: ERROR - REAL DATA REQUIRED! ==========
if data_source is None:
    print("\n❌ ERROR: NO REAL DATA AVAILABLE!")
    print("=" * 80)
    print("⚠️  ANALYSIS REQUIRES REAL DATA FROM:")
    print("   • Kafka topics (weather-api, weather-rss) OR")
    print("   • HDFS buffer (/data/weather/api, /data/weather/rss)")
    print("\n🔧 SOLUTION: Please start the data pipeline:")
    print("   1. Terminal 1: python kafka/producer_api.py")
    print("   2. Terminal 2: python kafka/producer_rss.py")
    print("   3. Terminal 3: python kafka/consumer_to_hdfs.py")
    print("   ⏳ Wait 2-3 minutes for data to populate HDFS")
    print("   ▶️  Then run this notebook again")
    print("=" * 80)
    raise Exception("⛔ NO REAL DATA - Analysis requires Kafka/HDFS data pipeline running!")

print(f"\n" + "="*80)
print(f"📊 DATA SUMMARY")
print(f"="*80)
print(f"  Source: {data_source.upper()}")
print(f"  Total Records: {len(df_api_pd) if df_api_pd is not None else 0}")
if df_api_pd is not None:
    print(f"  Columns: {', '.join(df_api_pd.columns.tolist())}")
    print(f"  Date Range: {df_api_pd['timestamp'].min()} to {df_api_pd['timestamp'].max()}")
    print(f"\n  First 5 rows:")
    print(df_api_pd.head(5).to_string(index=False))

📡 OPSI 1: Attempting to read from Kafka topics (weather-api, weather-rss)...
--------------------------------------------------------------------------------
  Reading from weather-api topic...
    ✓ Retrieved 12 records from weather-api
  Reading from weather-rss topic...
    ✓ Retrieved 105 records from weather-rss

  📊 Data Structure Debug:
     - API records: 12
     - RSS records: 105
     - API sample: ['kode_kota', 'nama_kota', 'latitude', 'longitude', 'temperature', 'humidity', 'wind_speed', 'weather_code', 'timestamp']
     - RSS sample keys: ['article_id', 'title', 'link', 'summary', 'published', 'source', 'timestamp']

✅ KAFKA DATA LOADED: 12 total records
   Sources: 12 API + 105 RSS

📊 DATA SUMMARY
  Source: KAFKA-LIVE
  Total Records: 12
  Columns: kode_kota, nama_kota, latitude, longitude, temperature, humidity, wind_speed, weather_code, timestamp
  Date Range: 2026-04-28T21:16:35.789532 to 2026-04-28T21:22:04.237412

  First 5 rows:
kode_kota nama_kota  latitude  longit

In [4]:
# ============================================================================
# ANALISIS 1: Temperature Comparison Across Cities
# ============================================================================
# Perbandingan suhu antar kota: rata-rata, tertinggi, terendah per kota
# Tujuan: Identifikasi kota mana yang paling panas/dingin

print("\n" + "="*80)
print("ANALISIS 1: PERBANDINGAN SUHU ANTAR KOTA")
print("="*80)

if not use_spark_mode:
    # ========== PANDAS MODE (Development) ==========
    print("\n📊 HASIL: Perbandingan Suhu per Kota (Pandas Mode)")
    print("-" * 80)
    
    # Group by city and calculate statistics
    temp_comparison_pd = df_api_pd.groupby(['kode_kota', 'nama_kota']).agg({
        'temperature': ['count', 'mean', 'max', 'min', 'std']
    }).round(2)
    
    temp_comparison_pd.columns = ['jumlah_observasi', 'rata_rata_suhu', 'suhu_tertinggi', 'suhu_terendah', 'std_dev_suhu']
    temp_comparison_pd['rentang_suhu'] = (temp_comparison_pd['suhu_tertinggi'] - temp_comparison_pd['suhu_terendah']).round(2)
    temp_comparison_pd = temp_comparison_pd.reset_index()
    temp_comparison_pd = temp_comparison_pd.sort_values('rata_rata_suhu', ascending=False)
    
    print(temp_comparison_pd[['kode_kota', 'nama_kota', 'jumlah_observasi', 'rata_rata_suhu', 'suhu_tertinggi', 'suhu_terendah', 'rentang_suhu']].to_string(index=False))

else:
    # ========== PANDAS MODE (consistent with data load) ==========
    print("\n📊 HASIL: Perbandingan Suhu per Kota (Pandas Mode)")
    print("-" * 80)
    
    # Using pandas for consistency even when use_spark_mode=True
    temp_comparison_pd = df_api_pd.groupby(['kode_kota', 'nama_kota']).agg({
        'temperature': ['count', 'mean', 'max', 'min', 'std']
    }).round(2)
    
    temp_comparison_pd.columns = ['jumlah_observasi', 'rata_rata_suhu', 'suhu_tertinggi', 'suhu_terendah', 'std_dev_suhu']
    temp_comparison_pd['rentang_suhu'] = (temp_comparison_pd['suhu_tertinggi'] - temp_comparison_pd['suhu_terendah']).round(2)
    temp_comparison_pd = temp_comparison_pd.reset_index()
    temp_comparison_pd = temp_comparison_pd.sort_values('rata_rata_suhu', ascending=False)

print("\n💡 INTERPRETASI BISNIS:")
print("-" * 80)

if len(temp_comparison_pd) > 0:
    hottest_city = temp_comparison_pd.loc[temp_comparison_pd['rata_rata_suhu'].idxmax()]
    coldest_city = temp_comparison_pd.loc[temp_comparison_pd['rata_rata_suhu'].idxmin()]
    most_volatile = temp_comparison_pd.loc[temp_comparison_pd['rentang_suhu'].idxmax()]

    print(f"1. KOTA TERPANAS: {hottest_city['nama_kota']} ({hottest_city['kode_kota']})")
    print(f"   - Rata-rata suhu: {hottest_city['rata_rata_suhu']}°C")
    print(f"   - Risiko: Kondisi ekstrem (suhu tinggi)")
    print(f"   - Rekomendasi: Perlu AC/pendingin di kendaraan pengiriman")

    print(f"\n2. KOTA TERDINGIN: {coldest_city['nama_kota']} ({coldest_city['kode_kota']})")
    print(f"   - Rata-rata suhu: {coldest_city['rata_rata_suhu']}°C")
    print(f"   - Risiko: Lebih stabil, kondisi pengiriman lebih baik")

    print(f"\n3. KOTA DENGAN FLUKTUASI TERBESAR: {most_volatile['nama_kota']} ({most_volatile['kode_kota']})")
    print(f"   - Rentang suhu: {most_volatile['suhu_terendah']}°C - {most_volatile['suhu_tertinggi']}°C")
    print(f"   - Variabilitas (StdDev): {most_volatile['std_dev_suhu']}°C")
    print(f"   - Risiko: Suhu tidak stabil, perlu perhatian untuk barang sensitif")

    print(f"\n📌 KESIMPULAN:")
    print(f"   Hindari {hottest_city['nama_kota']} pada jam panas & perhatikan {most_volatile['nama_kota']} untuk barang sensitif")

    analysis_1_results = {
        "analisis": "temperature_comparison",
        "timestamp": datetime.now().isoformat(),
        "summary": {
            "hottest_city": hottest_city['nama_kota'],
            "hottest_city_code": hottest_city['kode_kota'],
            "hottest_avg_temp": float(hottest_city['rata_rata_suhu']),
            "coldest_city": coldest_city['nama_kota'],
            "coldest_city_code": coldest_city['kode_kota'],
            "coldest_avg_temp": float(coldest_city['rata_rata_suhu']),
            "most_volatile_city": most_volatile['nama_kota'],
            "most_volatile_range": float(most_volatile['rentang_suhu'])
        },
        "detailed_data": temp_comparison_pd.to_dict('records')
    }
else:
    print("❌ No data available for analysis")
    analysis_1_results = {"analisis": "temperature_comparison", "timestamp": datetime.now().isoformat(), "summary": {}}


ANALISIS 1: PERBANDINGAN SUHU ANTAR KOTA

📊 HASIL: Perbandingan Suhu per Kota (Pandas Mode)
--------------------------------------------------------------------------------
kode_kota nama_kota  jumlah_observasi  rata_rata_suhu  suhu_tertinggi  suhu_terendah  rentang_suhu
      SBY  Surabaya                 2            27.5            27.5           27.5           0.0
      JKT   Jakarta                 2            27.0            27.0           27.0           0.0
      MDN     Medan                 2            26.7            26.7           26.7           0.0
      MKS  Makassar                 2            26.6            26.6           26.6           0.0
      SMG  Semarang                 2            26.0            26.0           26.0           0.0
      DPS  Denpasar                 2            25.9            25.9           25.9           0.0

💡 INTERPRETASI BISNIS:
--------------------------------------------------------------------------------
1. KOTA TERPANAS: Surabaya (

In [5]:
# ============================================================================
# ANALISIS 2: Extreme Weather Detection
# ============================================================================
# Deteksi kondisi ekstrem: wind_speed > 40 km/h OR humidity > 90% OR temperature > 35°C
# Tujuan: Identifikasi kapan dan di mana kondisi cuaca berbahaya untuk pengiriman

print("\n" + "="*80)
print("ANALISIS 2: DETEKSI KONDISI CUACA EKSTREM")
print("="*80)

if not use_spark_mode:
    # ========== PANDAS MODE (Development) ==========
    print("\n⚠️ HASIL: Deteksi Kondisi Ekstrem (Pandas Mode)")
    print("-" * 80)
    
    # Filter untuk kondisi ekstrem
    extreme_mask = (df_api_pd['wind_speed'] > 40) | (df_api_pd['humidity'] > 90) | (df_api_pd['temperature'] > 35)
    extreme_df_pd = df_api_pd[extreme_mask].copy()
    extreme_count = len(extreme_df_pd)
    
    # Classify extreme type
    extreme_df_pd['tipe_ekstrem'] = 'NORMAL'
    extreme_df_pd.loc[extreme_df_pd['wind_speed'] > 40, 'tipe_ekstrem'] = 'ANGIN KUAT'
    extreme_df_pd.loc[extreme_df_pd['humidity'] > 90, 'tipe_ekstrem'] = 'KELEMBABAN TINGGI'
    extreme_df_pd.loc[extreme_df_pd['temperature'] > 35, 'tipe_ekstrem'] = 'SUHU EKSTREM'
    
    if extreme_count > 0:
        print(f"⚠️ TOTAL EVENT CUACA EKSTREM TERDETEKSI: {extreme_count}\n")
        
        print("📋 DETAIL EVENT EKSTREM (Sample - max 10 events):")
        print("-" * 80)
        print(extreme_df_pd.head(10)[['kode_kota', 'nama_kota', 'timestamp', 'temperature', 'humidity', 'wind_speed', 'tipe_ekstrem']].to_string(index=False))
        
        print("\n📊 STATISTIK KONDISI EKSTREM PER KOTA:")
        print("-" * 80)
        
        extreme_summary_pd = extreme_df_pd.groupby(['kode_kota', 'nama_kota']).agg({
            'wind_speed': 'max',
            'humidity': 'max',
            'temperature': 'max'
        }).reset_index()
        extreme_summary_pd.columns = ['kode_kota', 'nama_kota', 'max_wind_speed', 'max_humidity', 'max_suhu']
        extreme_summary_pd['total_ekstrem'] = extreme_df_pd.groupby(['kode_kota', 'nama_kota']).size().values
        extreme_summary_pd['ekstrem_angin'] = ((extreme_df_pd['wind_speed'] > 40).groupby([extreme_df_pd['kode_kota'], extreme_df_pd['nama_kota']]).sum()).values
        extreme_summary_pd['ekstrem_kelembaban'] = ((extreme_df_pd['humidity'] > 90).groupby([extreme_df_pd['kode_kota'], extreme_df_pd['nama_kota']]).sum()).values
        extreme_summary_pd['ekstrem_suhu'] = ((extreme_df_pd['temperature'] > 35).groupby([extreme_df_pd['kode_kota'], extreme_df_pd['nama_kota']]).sum()).values
        
        print(extreme_summary_pd[['kode_kota', 'nama_kota', 'total_ekstrem', 'ekstrem_angin', 'ekstrem_kelembaban', 'ekstrem_suhu']].to_string(index=False))
    else:
        print(f"\n✅ Kondisi cuaca NORMAL. Tidak ada event ekstrem terdeteksi.")
        extreme_summary_pd = pd.DataFrame()

else:
    # ========== PANDAS MODE (consistent with data load) ==========
    print("\n⚠️ HASIL: Deteksi Kondisi Ekstrem (Pandas Mode)")
    print("-" * 80)
    
    # Filter untuk kondisi ekstrem
    extreme_mask = (df_api_pd['wind_speed'] > 40) | (df_api_pd['humidity'] > 90) | (df_api_pd['temperature'] > 35)
    extreme_df_pd = df_api_pd[extreme_mask].copy()
    extreme_count = len(extreme_df_pd)
    
    if extreme_count > 0:
        extreme_summary_pd = extreme_df_pd.groupby(['kode_kota', 'nama_kota']).size().reset_index(name='total_ekstrem')
    else:
        extreme_summary_pd = pd.DataFrame()

print("\n💡 INTERPRETASI BISNIS:")
print("-" * 80)

if extreme_count > 0 and len(extreme_summary_pd) > 0:
    most_extreme_city = extreme_summary_pd.loc[extreme_summary_pd['total_ekstrem'].idxmax()]
    
    print(f"1. KOTA PALING BERBAHAYA: {most_extreme_city['nama_kota']} ({most_extreme_city['kode_kota']})")
    print(f"   - Event ekstrem: {int(most_extreme_city['total_ekstrem'])} kali")
    print(f"   ⚠️  REKOMENDASI: HINDARI atau TUNDA pengiriman ke/dari {most_extreme_city['nama_kota']}")
    
    print(f"\n2. TIPE EKSTREM PALING SERING:")
    if 'ekstrem_angin' in extreme_summary_pd.columns:
        total_angin = extreme_summary_pd['ekstrem_angin'].sum()
        total_kelembaban = extreme_summary_pd['ekstrem_kelembaban'].sum()
        total_suhu = extreme_summary_pd['ekstrem_suhu'].sum()
        extreme_types = {
            'Angin Kuat': total_angin,
            'Kelembaban Tinggi': total_kelembaban,
            'Suhu Ekstrem': total_suhu
        }
        most_common = max(extreme_types, key=extreme_types.get)
        print(f"   - {most_common} ({extreme_types[most_common]} kejadian)")
    
    print(f"\n3. RISIKO LOGISTIK:")
    print(f"   - Angin kuat: Berisiko untuk pengiriman barang besar/ringan")
    print(f"   - Kelembaban tinggi: Berisiko untuk barang elektronik/tekstil")
    print(f"   - Suhu ekstrem: Berisiko untuk barang makanan/vaksin")
else:
    print("✅ Kondisi cuaca NORMAL. Tidak ada event ekstrem terdeteksi.")

analysis_2_results = {
    "analisis": "extreme_weather_detection",
    "timestamp": datetime.now().isoformat(),
    "summary": {
        "total_extreme_events": extreme_count,
        "dangerous_cities": extreme_summary_pd[['kode_kota', 'nama_kota', 'total_ekstrem']].to_dict('records') if len(extreme_summary_pd) > 0 else []
    }
}


ANALISIS 2: DETEKSI KONDISI CUACA EKSTREM

⚠️ HASIL: Deteksi Kondisi Ekstrem (Pandas Mode)
--------------------------------------------------------------------------------
⚠️ TOTAL EVENT CUACA EKSTREM TERDETEKSI: 2

📋 DETAIL EVENT EKSTREM (Sample - max 10 events):
--------------------------------------------------------------------------------
kode_kota nama_kota                  timestamp  temperature  humidity  wind_speed      tipe_ekstrem
      DPS  Denpasar 2026-04-28T21:16:39.982682         25.9        93         3.6 KELEMBABAN TINGGI
      DPS  Denpasar 2026-04-28T21:22:04.237412         25.9        93         3.6 KELEMBABAN TINGGI

📊 STATISTIK KONDISI EKSTREM PER KOTA:
--------------------------------------------------------------------------------
kode_kota nama_kota  total_ekstrem  ekstrem_angin  ekstrem_kelembaban  ekstrem_suhu
      DPS  Denpasar              2              0                   2             0

💡 INTERPRETASI BISNIS:
-----------------------------------------

In [6]:
# ============================================================================
# ANALISIS 3: Hourly Temperature Trend Analysis (Diurnal Pattern)
# ============================================================================
# Tren suhu per jam: rata-rata suhu semua kota per jam dalam sehari
# Tujuan: Identifikasi jam optimal untuk pengiriman berdasarkan pola suhu harian

print("\n" + "="*80)
print("ANALISIS 3: TREN SUHU PER JAM (POLA DIURNAL)")
print("="*80)

if not use_spark_mode:
    # ========== PANDAS MODE (Development) ==========
    print("\n📊 HASIL: Tren Suhu Rata-rata per Jam (Pandas Mode)")
    print("-" * 80)
    
    # Extract hour from timestamp
    df_api_pd['jam'] = pd.to_datetime(df_api_pd['timestamp']).dt.hour
    
    # Group by hour
    hourly_trend_pd = df_api_pd.groupby('jam').agg({
        'temperature': ['count', 'mean', 'max', 'min', 'std']
    }).round(2)
    
    hourly_trend_pd.columns = ['jumlah_observasi', 'rata_rata_suhu_jam', 'suhu_tertinggi_jam', 'suhu_terendah_jam', 'volatilitas_suhu']
    hourly_trend_pd = hourly_trend_pd.reset_index()
    hourly_trend_pd = hourly_trend_pd.sort_values('jam')
    
    print(hourly_trend_pd[['jam', 'jumlah_observasi', 'rata_rata_suhu_jam', 'suhu_tertinggi_jam', 'suhu_terendah_jam', 'volatilitas_suhu']].to_string(index=False))

else:
    # ========== PANDAS MODE (consistent with data load) ==========
    print("\n📊 HASIL: Tren Suhu Rata-rata per Jam (Pandas Mode)")
    print("-" * 80)
    
    # Extract hour from timestamp
    df_api_pd['jam'] = pd.to_datetime(df_api_pd['timestamp']).dt.hour
    
    # Group by hour
    hourly_trend_pd = df_api_pd.groupby('jam').agg({
        'temperature': ['count', 'mean', 'max', 'min', 'std']
    }).round(2)
    
    hourly_trend_pd.columns = ['jumlah_observasi', 'rata_rata_suhu_jam', 'suhu_tertinggi_jam', 'suhu_terendah_jam', 'volatilitas_suhu']
    hourly_trend_pd = hourly_trend_pd.reset_index()
    hourly_trend_pd = hourly_trend_pd.sort_values('jam')

print("\n💡 INTERPRETASI BISNIS - POLA DIURNAL (Pola Harian Suhu):")
print("-" * 80)

if len(hourly_trend_pd) > 0:
    hottest_hour = hourly_trend_pd.loc[hourly_trend_pd['rata_rata_suhu_jam'].idxmax()]
    coldest_hour = hourly_trend_pd.loc[hourly_trend_pd['rata_rata_suhu_jam'].idxmin()]
    most_volatile_hour = hourly_trend_pd.loc[hourly_trend_pd['volatilitas_suhu'].idxmax()]
    
    print(f"1. JAM TERPANAS: {int(hottest_hour['jam']):02d}:00 - {int(hottest_hour['jam']+1):02d}:00")
    print(f"   - Rata-rata suhu: {hottest_hour['rata_rata_suhu_jam']}°C")
    print(f"   - Rentang: {hottest_hour['suhu_terendah_jam']}°C - {hottest_hour['suhu_tertinggi_jam']}°C")
    print(f"   ❌ JANGAN pengiriman pada jam ini")
    
    print(f"\n2. JAM TERDINGIN: {int(coldest_hour['jam']):02d}:00 - {int(coldest_hour['jam']+1):02d}:00")
    print(f"   - Rata-rata suhu: {coldest_hour['rata_rata_suhu_jam']}°C")
    print(f"   ✅ IDEAL untuk pengiriman")
    
    print(f"\n3. JAM PALING TIDAK STABIL: {int(most_volatile_hour['jam']):02d}:00")
    print(f"   - StdDev: {most_volatile_hour['volatilitas_suhu']}°C")
    print(f"   ⚠️  Hindari pengiriman barang sensitif")
    
    # Rekomendasi jam pengiriman
    print(f"\n4. REKOMENDASI JAM PENGIRIMAN OPTIMAL:")
    print("-" * 80)
    
    median_temp = hourly_trend_pd['rata_rata_suhu_jam'].median()
    median_volatility = hourly_trend_pd['volatilitas_suhu'].median()
    good_hours = hourly_trend_pd[
        (hourly_trend_pd['rata_rata_suhu_jam'] < median_temp) &
        (hourly_trend_pd['volatilitas_suhu'] < median_volatility)
    ].sort_values('rata_rata_suhu_jam')
    
    if len(good_hours) > 0:
        print("   Jam-jam terbaik untuk pengiriman (suhu rendah + stabil):")
        for idx, row in good_hours.head(5).iterrows():
            print(f"   - {int(row['jam']):02d}:00: {row['rata_rata_suhu_jam']}°C (StdDev: {row['volatilitas_suhu']}°C)")
    
    print(f"\n5. POLA DIURNAL UMUM:")
    morning = hourly_trend_pd[hourly_trend_pd['jam'].between(6, 11)]['rata_rata_suhu_jam'].mean()
    afternoon = hourly_trend_pd[hourly_trend_pd['jam'].between(12, 17)]['rata_rata_suhu_jam'].mean()
    evening = hourly_trend_pd[hourly_trend_pd['jam'].between(18, 23)]['rata_rata_suhu_jam'].mean()
    night = hourly_trend_pd[hourly_trend_pd['jam'].between(0, 5)]['rata_rata_suhu_jam'].mean()
    
    print(f"   - Malam (00-05): {round(night, 2)}°C (PALING DINGIN)")
    print(f"   - Pagi (06-11): {round(morning, 2)}°C")
    print(f"   - Siang (12-17): {round(afternoon, 2)}°C (PALING PANAS)")
    print(f"   - Sore (18-23): {round(evening, 2)}°C")
    print(f"\n   📌 Pengiriman lebih baik di malam/pagi saat suhu rendah & stabil")

else:
    print("   ⚠️  Data tidak tersedia untuk analisis jam-per-jam")
    good_hours = pd.DataFrame()

# Simpan hasil
analysis_3_results = {
    "analisis": "hourly_temperature_trend",
    "timestamp": datetime.now().isoformat(),
    "summary": {
        "hottest_hour": int(hottest_hour['jam']) if len(hourly_trend_pd) > 0 else None,
        "coldest_hour": int(coldest_hour['jam']) if len(hourly_trend_pd) > 0 else None,
        "optimal_delivery_hours": hourly_trend_pd[hourly_trend_pd['jam'].isin(good_hours['jam'])]['jam'].tolist() if len(good_hours) > 0 else []
    },
    "hourly_data": hourly_trend_pd.to_dict('records')
}


ANALISIS 3: TREN SUHU PER JAM (POLA DIURNAL)

📊 HASIL: Tren Suhu Rata-rata per Jam (Pandas Mode)
--------------------------------------------------------------------------------
 jam  jumlah_observasi  rata_rata_suhu_jam  suhu_tertinggi_jam  suhu_terendah_jam  volatilitas_suhu
  21                12               26.62                27.5               25.9              0.58

💡 INTERPRETASI BISNIS - POLA DIURNAL (Pola Harian Suhu):
--------------------------------------------------------------------------------
1. JAM TERPANAS: 21:00 - 22:00
   - Rata-rata suhu: 26.62°C
   - Rentang: 25.9°C - 27.5°C
   ❌ JANGAN pengiriman pada jam ini

2. JAM TERDINGIN: 21:00 - 22:00
   - Rata-rata suhu: 26.62°C
   ✅ IDEAL untuk pengiriman

3. JAM PALING TIDAK STABIL: 21:00
   - StdDev: 0.58°C
   ⚠️  Hindari pengiriman barang sensitif

4. REKOMENDASI JAM PENGIRIMAN OPTIMAL:
--------------------------------------------------------------------------------

5. POLA DIURNAL UMUM:
   - Malam (00-05): nan°C

In [7]:
# ============================================================================
# SECTION 4: Save Results to Local JSON and Optional HDFS
# ============================================================================
# Simpan semua hasil analisis untuk dashboard (local mode) dan HDFS (production)

print("\n" + "="*80)
print("MENYIMPAN HASIL ANALISIS")
print("="*80)

# Gabung semua hasil analisis
# Helper function untuk handle NaN/Infinity values
def convert_for_json(obj):
    if isinstance(obj, float):
        if obj != obj:  # NaN check
            return None
        if obj == float('inf'):
            return "Infinity"
        if obj == float('-inf'):
            return "-Infinity"
    return obj

all_results = {
    "metadata": {
        "project": "WeatherPulse",
        "analysis_timestamp": datetime.now().isoformat(),
        "description": "Spark analysis untuk 6 kota besar Indonesia",
        "cities": ["Jakarta", "Surabaya", "Semarang", "Medan", "Makassar", "Denpasar"],
        "environment": "hdfs-production" if use_spark_mode else "kafka-production" if data_source == "kafka-live" else "unknown",
        "mode": "pandas-development" if not use_spark_mode else "spark-production"
    },
    "analysis": {
        "analisis_1_temperature_comparison": analysis_1_results,
        "analisis_2_extreme_weather": analysis_2_results,
        "analisis_3_hourly_trend": analysis_3_results
    }
}

# ========== SAVE TO LOCAL JSON (untuk dashboard development) ==========
print("\n1️⃣  Saving to Local JSON (for Dashboard):")

dashboard_data_dir = "dashboard/data"
os.makedirs(dashboard_data_dir, exist_ok=True)

spark_results_path = os.path.join(dashboard_data_dir, "spark_results.json")

try:
    with open(spark_results_path, 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"   ✓ {spark_results_path}")
    print(f"     Size: {os.path.getsize(spark_results_path) / 1024:.1f} KB")
except Exception as e:
    print(f"   ✗ Error: {e}")

# ========== SAVE TO HDFS (opsional, hanya jika HDFS available) ==========
print("\n2️⃣  Saving to HDFS (Optional):")

if use_spark_mode and data_source == "hdfs-consumer":
    hdfs_results_path = "hdfs://namenode:8020/data/weather/hasil"
    
    try:
        print("   Attempting to save to HDFS...")
        
        # Convert pandas to Spark DataFrame jika perlu
        try:
            if 'temp_comparison_pd' in locals() and temp_comparison_pd is not None:
                df_temp_comparison = spark.createDataFrame(temp_comparison_pd)
                df_temp_comparison.coalesce(1).write \
                    .mode("overwrite") \
                    .json(f"{hdfs_results_path}/analisis_1_temperature_comparison")
                print(f"   ✓ Analisis 1")
        except Exception as e:
            print(f"   ⚠ Analisis 1 skip: {type(e).__name__}")
        
        # Analisis 2
        try:
            if extreme_count > 0 and 'extreme_df_pd' in locals() and extreme_df_pd is not None:
                df_extreme = spark.createDataFrame(extreme_df_pd)
                df_extreme.coalesce(1).write \
                    .mode("overwrite") \
                    .json(f"{hdfs_results_path}/analisis_2_extreme_weather")
                print(f"   ✓ Analisis 2")
        except Exception as e:
            print(f"   ⚠ Analisis 2 skip: {type(e).__name__}")
        
        # Analisis 3
        try:
            if 'hourly_trend_pd' in locals() and hourly_trend_pd is not None:
                df_hourly_trend = spark.createDataFrame(hourly_trend_pd)
                df_hourly_trend.coalesce(1).write \
                    .mode("overwrite") \
                    .json(f"{hdfs_results_path}/analisis_3_hourly_trend")
                print(f"   ✓ Analisis 3")
        except Exception as e:
            print(f"   ⚠ Analisis 3 skip: {type(e).__name__}")
        
    except Exception as e:
        print(f"   ✗ Could not save to HDFS: {type(e).__name__}")
else:
    print("   ⏭ Skipped (using development mode)")
    print("   ℹ Data saved only to local JSON for dashboard")

print(f"\n✅ Analysis complete!")
print(f"   📊 Dashboard data: {spark_results_path}")
print(f"   📁 Mode: {'Pandas (Development)' if not use_spark_mode else 'Spark SQL (Production)'}")


MENYIMPAN HASIL ANALISIS

1️⃣  Saving to Local JSON (for Dashboard):
   ✓ dashboard/data\spark_results.json
     Size: 3.5 KB

2️⃣  Saving to HDFS (Optional):
   ⏭ Skipped (using development mode)
   ℹ Data saved only to local JSON for dashboard

✅ Analysis complete!
   📊 Dashboard data: dashboard/data\spark_results.json
   📁 Mode: Pandas (Development)


## Ringkasan Eksekutif: Rekomendasi untuk Klien Logistik

### 🎯 Pertanyaan Bisnis yang Dijawab
**"Kota mana yang kondisi cuacanya paling ekstrem hari ini, dan kapan perkiraan terbaik untuk pengiriman?"**

### 📌 Kesimpulan Analisis

Berdasarkan 3 analisis Spark yang telah dijalankan, berikut adalah rekomendasi untuk perusahaan logistik:

#### **ANALISIS 1: Perbandingan Suhu Antar Kota**
- Identifikasi kota dengan suhu tertinggi dan terendah
- Perhatian khusus untuk kota dengan fluktuasi suhu besar (volatile)
- **Rekomendasi:** Prioritaskan pengiriman ke kota dengan suhu stabil

#### **ANALISIS 2: Deteksi Kondisi Ekstrem**
- Mengidentifikasi event cuaca berbahaya (angin kuat, kelembaban tinggi, suhu ekstrem)
- Pemetaan risiko per kota
- **Rekomendasi:** HINDARI pengiriman ke kota yang sedang mengalami kondisi ekstrem

#### **ANALISIS 3: Pola Diurnal (Tren Suhu Harian)**
- Identifikasi jam-jam optimal untuk pengiriman berdasarkan suhu
- Pola umum: malam/pagi lebih dingin, siang paling panas
- **Rekomendasi:** Jadwalkan pengiriman saat suhu rendah dan stabil (biasanya malam/pagi)

---

### 💼 Business Impact
✅ Mengurangi risiko kerusakan barang akibat cuaca ekstrem  
✅ Optimasi jadwal pengiriman untuk efisiensi dan keselamatan  
✅ Monitoring real-time untuk pengambilan keputusan cepat  
✅ Data-driven logistics planning untuk 6 kota besar Indonesia  

---

### 📊 Dashboard Connection
Hasil analisis ini telah disimpan dalam format JSON untuk ditampilkan di dashboard Flask:
- Tabel perbandingan suhu per kota (dengan warna indikator)
- Highlight kota dengan kondisi ekstrem
- Grafik tren suhu per jam untuk identifikasi waktu optimal pengiriman

In [11]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend untuk stability
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# BONUS: Data Visualization untuk Dashboard
# ============================================================================
# Visualisasi hasil analisis menggunakan matplotlib

print("\n" + "="*80)
print("BONUS: VISUALISASI DATA (Untuk Dashboard)")
print("="*80)

# Set style
sns.set_style("whitegrid")

try:
    # Buat figure dengan 4 subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('WeatherPulse Analytics Dashboard', fontsize=16, fontweight='bold')

    # Chart 1: Temperature Comparison Bar Chart
    if len(temp_comparison_pd) > 0:
        ax1 = axes[0, 0]
        colors = ['red' if x > 30 else 'orange' if x > 28 else 'yellow' for x in temp_comparison_pd['rata_rata_suhu']]
        ax1.bar(temp_comparison_pd['kode_kota'], temp_comparison_pd['rata_rata_suhu'], color=colors, alpha=0.7)
        ax1.set_title('Rata-rata Suhu per Kota', fontweight='bold')
        ax1.set_ylabel('Suhu (°C)')
        ax1.set_xlabel('Kota')
        ax1.axhline(y=30, color='red', linestyle='--', linewidth=1, label='Threshold Ekstrem')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    else:
        ax1 = axes[0, 0]
        ax1.text(0.5, 0.5, 'No temperature data', ha='center', va='center', transform=ax1.transAxes)
        ax1.set_title('Rata-rata Suhu per Kota', fontweight='bold')

    # Chart 2: Temperature Range Volatility
    if len(temp_comparison_pd) > 0:
        ax2 = axes[0, 1]
        ax2.barh(temp_comparison_pd['kode_kota'], temp_comparison_pd['rentang_suhu'], color='steelblue', alpha=0.7)
        ax2.set_title('Rentang Suhu per Kota (Volatilitas)', fontweight='bold')
        ax2.set_xlabel('Rentang Suhu (°C)')
        ax2.grid(True, alpha=0.3)
    else:
        ax2 = axes[0, 1]
        ax2.text(0.5, 0.5, 'No volatility data', ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title('Rentang Suhu per Kota (Volatilitas)', fontweight='bold')

    # Chart 3: Hourly Temperature Trend
    if len(hourly_trend_pd) > 0:
        ax3 = axes[1, 0]
        ax3.plot(hourly_trend_pd['jam'], hourly_trend_pd['rata_rata_suhu_jam'], 
                 marker='o', linewidth=2, markersize=6, color='#FF6B6B', label='Rata-rata')
        ax3.fill_between(hourly_trend_pd['jam'], 
                          hourly_trend_pd['suhu_terendah_jam'], 
                          hourly_trend_pd['suhu_tertinggi_jam'], 
                          alpha=0.3, color='#FF6B6B', label='Range Min-Max')
        ax3.set_title('Tren Suhu Berdasarkan Jam (Pola Diurnal)', fontweight='bold')
        ax3.set_xlabel('Jam dalam Sehari')
        ax3.set_ylabel('Suhu (°C)')
        ax3.set_xticks(range(0, 24, 2))
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    else:
        ax3 = axes[1, 0]
        ax3.text(0.5, 0.5, 'No hourly data', ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title('Tren Suhu Berdasarkan Jam (Pola Diurnal)', fontweight='bold')

    # Chart 4: Extreme Weather Count
    if len(extreme_summary_pd) > 0 and extreme_count > 0:
        ax4 = axes[1, 1]
        colors_extreme = ['darkred' if x > 5 else 'orange' for x in extreme_summary_pd['total_ekstrem']]
        ax4.bar(extreme_summary_pd['kode_kota'], extreme_summary_pd['total_ekstrem'], 
                color=colors_extreme, alpha=0.7)
        ax4.set_title('Jumlah Event Cuaca Ekstrem per Kota', fontweight='bold')
        ax4.set_ylabel('Jumlah Event')
        ax4.set_xlabel('Kota')
        ax4.grid(True, alpha=0.3)
    else:
        ax4 = axes[1, 1]
        ax4.text(0.5, 0.5, 'Tidak ada data ekstrem\n(Semua kota kondisi normal)', 
                ha='center', va='center', fontsize=12, transform=ax4.transAxes)
        ax4.set_title('Jumlah Event Cuaca Ekstrem per Kota', fontweight='bold')

    plt.tight_layout()

    # Simpan visualization ke file
    viz_path = os.path.join(dashboard_data_dir, "weather_analysis_charts.png")
    plt.savefig(viz_path, dpi=100, bbox_inches='tight')
    print(f"✓ Chart saved: {viz_path}")
    try:
        print(f"  Size: {os.path.getsize(viz_path) / 1024:.1f} KB")
    except:
        pass
    
    # Close plot untuk free memory
    plt.close(fig)

except Exception as e:
    print(f"⚠ Error creating visualization: {type(e).__name__}")
    print(f"  {str(e)[:100]}")
    print("  Continuing without visualization...")

print("\n✅ Bonus visualization complete!")


BONUS: VISUALISASI DATA (Untuk Dashboard)
⚠ Error creating visualization: NameError
  name 'dashboard_data_dir' is not defined
  Continuing without visualization...

✅ Bonus visualization complete!
